# Deep CNN Image Classification

This notebook implements and trains the Deep Convolutional Neural Network (Deep CNN)
for six-class natural scene image classification using the Intel Image Classification Dataset.

## Model Configuration

- Input Size: 224 × 224 × 3
- Number of Classes: 6
- Activation Function: ReLU
- Output Activation: Softmax
- Loss Function: Sparse Categorical Crossentropy
- Optimizer: Adam
- Learning Rate: 0.001
- Batch Size: 32
- Maximum Epochs: 30
- Dropout Rate: 0.5

In [3]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [6]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models

In [11]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

train_path = "../dataset/Dataset/train"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Classes:", train_ds.class_names)

Found 12632 files belonging to 6 classes.
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [10]:
import os

print(os.getcwd())

d:\4th Year\Modules\SE4050 - DL\DL Assignment\deep-learning-image-classification\notebooks


In [12]:
val_path = "../dataset/Dataset/val"

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Validation classes:", val_ds.class_names)

Found 1402 files belonging to 6 classes.
Validation classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [13]:
test_path = "../dataset/Dataset/test"

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Test classes:", test_ds.class_names)

Found 3000 files belonging to 6 classes.
Test classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [14]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(
    lambda images, labels: (normalization_layer(images), labels)
)

val_ds = val_ds.map(
    lambda images, labels: (normalization_layer(images), labels)
)

test_ds = test_ds.map(
    lambda images, labels: (normalization_layer(images), labels)
)

In [15]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    )
])

In [16]:
def build_deep_cnn():
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),

        layers.Dense(6, activation="softmax")
    ])

    return model

In [17]:
model = build_deep_cnn()

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 422,086 (1.61 MB)

 Trainable params: 422,086 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
train_ds_augmented = train_ds.map(
    lambda images, labels: (data_augmentation(images, training=True), labels)
)

In [20]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "../models/deep_cnn_best.keras",
    monitor="val_loss",
    save_best_only=True
)

In [21]:
EPOCHS = 30

print("Training configuration:")
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", 0.001)

Training configuration:
Epochs: 30
Batch size: 32
Learning rate: 0.001


In [22]:
history = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        early_stopping,
        model_checkpoint
    ]
)

Epoch 1/30


C:\Users\Mineth\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


395/395 ━━━━━━━━━━━━━━━━━━━━ 301s 754ms/step - accuracy: 0.4279 - loss: 1.3254 - val_accuracy: 0.5271 - val_loss: 1.1153
Epoch 2/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 277s 701ms/step - accuracy: 0.5239 - loss: 1.1447 - val_accuracy: 0.5984 - val_loss: 0.9828
Epoch 3/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 275s 695ms/step - accuracy: 0.5935 - loss: 1.0288 - val_accuracy: 0.6491 - val_loss: 0.8913
Epoch 4/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 275s 696ms/step - accuracy: 0.6453 - loss: 0.9210 - val_accuracy: 0.6940 - val_loss: 0.8075
Epoch 5/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 281s 712ms/step - accuracy: 0.6841 - loss: 0.8420 - val_accuracy: 0.7389 - val_loss: 0.7200
Epoch 6/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 290s 733ms/step - accuracy: 0.7097 - loss: 0.7867 - val_accuracy: 0.7675 - val_loss: 0.6374
Epoch 7/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 343s 866ms/step - accuracy: 0.7341 - loss: 0.7450 - val_accuracy: 0.7896 - val_loss: 0.5942
Epoch 8/30
395/395 ━━━━━━━━━━━━━━━━━━━━ 2244s 6s/step - accuracy: 0.7418 - loss: 0.7156

In [23]:
best_model = tf.keras.models.load_model("../models/deep_cnn_best.keras")

print("Best Deep CNN model loaded successfully.")

Best Deep CNN model loaded successfully.


In [24]:
test_loss, test_accuracy = best_model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

94/94 ━━━━━━━━━━━━━━━━━━━━ 17s 173ms/step - accuracy: 0.8543 - loss: 0.4173
Test Loss: 0.41729623079299927
Test Accuracy: 0.8543333411216736
